# Shark Knowledge Medallion — Overview

Base de conocimiento especializada sobre tiburones, construida con una **Arquitectura Medallón** (Bronze → Silver → Merge → Warehouse → Gold) sobre datos de GBIF y Wikipedia.

---

## 1. Planteamiento del proyecto

### Objetivo

Construir una base de conocimiento canónica, determinista y completamente local sobre especies de tiburones, integrando:

- **GBIF** (Global Biodiversity Information Facility) — taxonomía y registros de ocurrencia georreferenciados.
- **Wikipedia en inglés** — descripciones textuales completas por especie.

El sistema organiza los datos en cinco etapas (Arquitectura Medallón), desde el JSON crudo hasta un índice vectorial listo para búsqueda semántica híbrida.

### Principios

- Pipeline determinista, local-first, idempotente y auditable.
- Clave natural de especie: `species_key = slugify(scientific_name)`.
- Universo de especies abierto: el lote semilla es solo un punto de partida.

In [1]:
import os
from pathlib import Path

# Ubicar raíz del repo (donde está src/) y posicionarse ahí
while not (Path.cwd() / "src").exists():
    os.chdir("..")
    if Path.cwd().parent == Path.cwd():
        raise RuntimeError("No se encontró el directorio src/ — ejecuta el notebook desde el repo")

In [2]:
from src.config.seed_species import SEED_SPECIES
from src.resolver.slugify import slugify
from src.pipeline import run_pipeline

/home/luisgm/Documents/Diplomado AI Engineer - Augemented Humans/Mis repos/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print(f'Lote semilla: {len(SEED_SPECIES)} especies\n')
for s in SEED_SPECIES:
    print(f'  {s["species_key"]:35s} | {s["scientific_name"]:30s} | {s["wikipedia_title"]}')

Lote semilla: 18 especies

  carcharodon_carcharias              | Carcharodon carcharias         | Great white shark
  rhincodon_typus                     | Rhincodon typus                | Whale shark
  cetorhinus_maximus                  | Cetorhinus maximus             | Basking shark
  galeocerdo_cuvier                   | Galeocerdo cuvier              | Tiger shark
  sphyrna_mokarran                    | Sphyrna mokarran               | Great hammerhead
  sphyrna_lewini                      | Sphyrna lewini                 | Scalloped hammerhead
  prionace_glauca                     | Prionace glauca                | Blue shark
  isurus_oxyrinchus                   | Isurus oxyrinchus              | Shortfin mako shark
  carcharhinus_leucas                 | Carcharhinus leucas            | Bull shark
  carcharhinus_longimanus             | Carcharhinus longimanus        | Oceanic whitetip shark
  triaenodon_obesus                   | Triaenodon obesus              | Whitetip re

In [4]:
# species_key: identificador canónico determinista
nombres = ['Prionace glauca', 'Carcharodon carcharias', 'Rhincodon typus']
for n in nombres:
    print(f'  {n:30s} -> {slugify(n)}')

  Prionace glauca                -> prionace_glauca
  Carcharodon carcharias         -> carcharodon_carcharias
  Rhincodon typus                -> rhincodon_typus


In [5]:
# Firma del pipeline: orquesta Bronze -> Silver -> Merge -> Warehouse -> Gold
import inspect
print(inspect.signature(run_pipeline))

(species_names: 'list[str]', batch_id: 'str | None' = None) -> 'dict'


---
## 2. Origen de los datos

### APIs consumidas

| Fuente | Endpoint | Propósito |
|--------|----------|-----------|
| GBIF Species Match | `GET /v1/species/match?name={scientificName}` | Taxonomía: kingdom, phylum, class, rank |
| GBIF Occurrence Search | `GET /v1/occurrence/search?scientificName={nombre}&hasCoordinate=true&limit=50` | Ocurrencias: país, coordenadas, fecha, IUCN |
| Wikipedia REST | `GET /page/summary/{title}` | Resumen introductorio por especie |
| Wikipedia Action API | `GET /w/api.php?action=query&prop=extracts&explaintext=true&titles={title}` | Extractos completos por secciones |

Ninguna requiere autenticación. Wikipedia es la **fuente pivote** (descriptiva); GBIF es **fuente de augmentation** (taxonomía + geografía + conservación).

In [6]:
from src.bronze.fetcher import (
    GBIF_SPECIES_MATCH,
    GBIF_OCCURRENCE,
    WIKIPEDIA_API,
    WIKIPEDIA_REST,
    fetch_gbif_taxonomy,
    fetch_gbif_occurrences,
    fetch_wikipedia_summary,
    fetch_wikipedia_extracts,
    persist_bronze,
)
from src.resolver.wikipedia import resolve_wikipedia_title, ResolutionMethod, ResolutionStatus

In [7]:
print('Endpoints GBIF')
print(f'  Species Match:  {GBIF_SPECIES_MATCH}')
print(f'  Occurrence:     {GBIF_OCCURRENCE}')
print()
print('Endpoints Wikipedia')
print(f'  REST API:       {WIKIPEDIA_REST}/page/summary/...')
print(f'  Action API:     {WIKIPEDIA_API}')

Endpoints GBIF
  Species Match:  https://api.gbif.org/v1/species/match
  Occurrence:     https://api.gbif.org/v1/occurrence/search

Endpoints Wikipedia
  REST API:       https://en.wikipedia.org/api/rest_v1/page/summary/...
  Action API:     https://en.wikipedia.org/w/api.php


### 2.1 GBIF Species Match — Taxonomía exacta

Devuelve el `usageKey`, `rank`, `kingdom`, `phylum`, `class` y el `matchType`. Solo aceptamos `matchType == 'EXACT'`.

In [8]:
# Consultar taxonomía de una especie vía GBIF
tax_raw = fetch_gbif_taxonomy('Carcharodon carcharias')
print(f'HTTP Status: {tax_raw["status"]}\n')
print('Respuesta GBIF Species Match:')
import json
print(json.dumps(tax_raw['data'], indent=2))

HTTP Status: 200

Respuesta GBIF Species Match:
{
  "usageKey": 2420694,
  "scientificName": "Carcharodon carcharias (Linnaeus, 1758)",
  "canonicalName": "Carcharodon carcharias",
  "rank": "SPECIES",
  "status": "ACCEPTED",
  "confidence": 99,
  "matchType": "EXACT",
  "kingdom": "Animalia",
  "phylum": "Chordata",
  "order": "Lamniformes",
  "family": "Lamnidae",
  "genus": "Carcharodon",
  "species": "Carcharodon carcharias",
  "kingdomKey": 1,
  "phylumKey": 44,
  "classKey": 121,
  "orderKey": 885,
  "familyKey": 5888,
  "genusKey": 2420693,
  "speciesKey": 2420694,
  "class": "Elasmobranchii"
}


In [9]:
# Caso: especie que NO hace match exacto — será rechazada en Silver
tax_bad = fetch_gbif_taxonomy('Fakeus nonexistus')
print(f'HTTP Status: {tax_bad["status"]}')
resp = tax_bad['data']
if isinstance(resp, dict):
    print(f'matchType: {resp.get("matchType", "N/A")}')
    print(f'confidence: {resp.get("confidence", "N/A")}')
else:
    print(f'Response: {resp}')

HTTP Status: 200
matchType: NONE
confidence: 100


### 2.2 GBIF Occurrence Search — Registros de ocurrencia

Devuelve observaciones con coordenadas, país, fecha, categoría IUCN y `basisOfRecord`.

In [10]:
# Ocurrencias de tiburón ballena (Rhincodon typus)
occ_raw = fetch_gbif_occurrences('Rhincodon typus')
results = occ_raw['data'].get('results', [])
print(f'HTTP Status: {occ_raw["status"]}')
print(f'Total de ocurrencias devueltas: {len(results)}\n')
print('Primeras 2 ocurrencias:')
for r in results[:2]:
    print(json.dumps({
        'gbifID': r.get('gbifID'),
        'country': r.get('country'),
        'decimalLatitude': r.get('decimalLatitude'),
        'decimalLongitude': r.get('decimalLongitude'),
        'eventDate': r.get('eventDate'),
        'basisOfRecord': r.get('basisOfRecord'),
        'iucnRedListCategory': r.get('iucnRedListCategory'),
    }, indent=2))
    print()

HTTP Status: 200
Total de ocurrencias devueltas: 50

Primeras 2 ocurrencias:
{
  "gbifID": "5938116839",
  "country": "Maldives",
  "decimalLatitude": 3.717267,
  "decimalLongitude": 72.758961,
  "eventDate": "2026-01-03T11:02",
  "basisOfRecord": "HUMAN_OBSERVATION",
  "iucnRedListCategory": "EN"
}

{
  "gbifID": "5938382709",
  "country": "Philippines",
  "decimalLatitude": 9.98467,
  "decimalLongitude": 125.279289,
  "eventDate": "2026-01-03T09:28:03",
  "basisOfRecord": "HUMAN_OBSERVATION",
  "iucnRedListCategory": "EN"
}



### 2.3 Resolución de títulos de Wikipedia

El sistema resuelve automáticamente `scientific_name` → título de Wikipedia usando: (1) consulta directa, (2) redirects, (3) búsqueda por texto.

In [11]:
# Resolución directa
res = resolve_wikipedia_title('Great white shark')
print('Entrada:           Great white shark')
print(f'Título resuelto:   {res.title}')
print(f'Método:            {res.resolved_via}')
print(f'Status:            {res.status.value}')
print()

# Resolución vía nombre científico (search fallback)
res2 = resolve_wikipedia_title('Carcharodon carcharias')
print('Entrada:           Carcharodon carcharias')
print(f'Título resuelto:   {res2.title}')
print(f'Método:            {res2.resolved_via}')
print(f'Status:            {res2.status.value}')

Entrada:           Great white shark
Título resuelto:   Great white shark
Método:            ResolutionMethod.DIRECT_TITLE
Status:            SUCCESS

Entrada:           Carcharodon carcharias
Título resuelto:   Great white shark
Método:            ResolutionMethod.REDIRECT
Status:            SUCCESS


In [12]:
# Caso: especie sin página de Wikipedia -> PAGE_NOT_FOUND
res3 = resolve_wikipedia_title('Fakeus nonexistus')
print('Entrada:           Fakeus nonexistus')
print(f'Status:            {res3.status.value}')
print(f'Candidatos:        {res3.candidates}')

Entrada:           Fakeus nonexistus
Status:            PAGE_NOT_FOUND
Candidatos:        None


### 2.4 Wikipedia — Resumen y extractos completos

Dos llamadas por especie: `summary` (REST, párrafo introductorio) y `extracts` (Action API, todas las secciones del artículo).

In [13]:
# Resumen Wikipedia del tiburón blanco
wp_summary = fetch_wikipedia_summary('Great white shark')
print(f'HTTP Status: {wp_summary["status"]}')
data = wp_summary['data']
print(f'Título:    {data.get("title")}')
print(f'Extracto:  {data.get("extract", "")[:400]}...')

HTTP Status: 200
Título:    Great white shark
Extracto:  The great white shark, also known as the white shark, white pointer, or great white, is a large shark. It is closely related to the mako sharks, the porbeagle, and the salmon shark. It is a robustly built species with a grayish upperside and a white underside. The white shark is one of the largest living shark and fish species, though it is still smaller than the whale shark and basking shark. It ...


In [14]:
# Extractos completos (por secciones)
wp_extracts = fetch_wikipedia_extracts('Great white shark')
pages = wp_extracts['data'].get('query', {}).get('pages', {})
for pid, pdata in pages.items():
    extract_text = pdata.get('extract', '')
    print(f'Título: {pdata.get("title")}')
    print(f'Longitud del extracto: {len(extract_text)} caracteres')
    print()
    # Mostrar las primeras líneas para ver las secciones
    for line in extract_text.split('\n')[:100]:
        if line.startswith('=='):
            print(f'  SECCIÓN: {line.strip()}')
    print('  ...')

Título: Great white shark
Longitud del extracto: 47182 caracteres

  SECCIÓN: == Etymology and naming ==
  SECCIÓN: == Taxonomy and evolution ==
  SECCIÓN: === Populations and genetic history ===
  SECCIÓN: == Appearance and anatomy ==
  SECCIÓN: === Size ===
  SECCIÓN: === Teeth and jaws ===
  SECCIÓN: === Senses ===
  SECCIÓN: === Internal physiology ===
  SECCIÓN: == Distribution and habitat ==
  SECCIÓN: === Migrations ===
  SECCIÓN: == Behavior and ecology ==
  SECCIÓN: === Diet and feeding ===
  SECCIÓN: === Social communication ===
  SECCIÓN: === Reproduction and growth ===
  SECCIÓN: === Mortality and health ===
  ...


### 2.5 Bronze — Persistencia de datos crudos

Todo se persiste como JSON crudo timestamped en `data/bronze/{source}/`, nada se descarta ni transforma.

In [15]:
# Demostración del envelope de metadatos que usa persist_bronze
gbif_sample = fetch_gbif_taxonomy('Galeocerdo cuvier')
path = persist_bronze(
    source='gbif_taxonomy',
    lote=1,
    species_queried=['Galeocerdo cuvier'],
    raw_responses=[{'query': 'Galeocerdo cuvier', 'response': gbif_sample}],
)
print(f'Archivo persistido: {path}')
with open(path) as f:
    envelope = json.load(f)
print('\nEnvelope:')
print(f'  source:         {envelope["source"]}')
print(f'  lote:           {envelope["lote"]}')
print(f'  ingested_at:    {envelope["ingested_at"]}')
print(f'  species_queried:{envelope["species_queried"]}')

Archivo persistido: data/bronze/gbif_taxonomy/lote_1__2026-07-28T00-55-54Z.json

Envelope:
  source:         gbif_taxonomy
  lote:           1
  ingested_at:    2026-07-28T00:55:54Z
  species_queried:['Galeocerdo cuvier']


---
## 3. Arquitectura Medallón

```
GBIF Species Match ──┐
GBIF Occurrence ─────┼──► BRONZE (JSON crudo, timestamped)
Wikipedia (en) ──────┘
            │
            ▼
SILVER (contratos Pydantic independientes por fuente
        + granularidad propia: especie, ocurrencia o página)
            │
            ▼
MERGE (Wikipedia = pivote 1:1, GBIF = augmentation agregada)
            │
            ▼
WAREHOUSE (DuckDB, UPSERT idempotente por clave natural)
            │
            ▼
GOLD (Species / SpeciesChunk / SpeciesMetadata, FAISS IndexHNSWFlat)
```

### 3.1 Bronze Layer — Raw JSON

Propósito: Capturar la respuesta íntegra de cada API en archivos JSON con timestamp, **sin transformar ni filtrar**. Incluso errores HTTP se persisten tal cual.

In [16]:
from src.bronze.fetcher import fetch_gbif_taxonomy, fetch_gbif_occurrences, fetch_wikipedia_summary, fetch_wikipedia_extracts, persist_bronze

print('Funciones de ingesta:')
print('  fetch_gbif_taxonomy(scientific_name)    -> dict {status, data}')
print('  fetch_gbif_occurrences(scientific_name) -> dict {status, data}')
print('  fetch_wikipedia_summary(title)          -> dict {status, data}')
print('  fetch_wikipedia_extracts(title)         -> dict {status, data}')
print('  persist_bronze(source, lote, species_queried, raw_responses) -> str (ruta)')

Funciones de ingesta:
  fetch_gbif_taxonomy(scientific_name)    -> dict {status, data}
  fetch_gbif_occurrences(scientific_name) -> dict {status, data}
  fetch_wikipedia_summary(title)          -> dict {status, data}
  fetch_wikipedia_extracts(title)         -> dict {status, data}
  persist_bronze(source, lote, species_queried, raw_responses) -> str (ruta)


### 3.2 Silver Layer — Validación y contratos

Cada fuente tiene un **contrato Pydantic v2** independiente. Los registros que no cumplen se rechazan con motivo explícito y se aíslan en `data/silver/{fuente}/rejected/`.

In [17]:
from src.silver.models import WikipediaPage, GBIFTaxonomy, GBIFOccurrence
from src.silver.validator import validate_wikipedia, validate_gbif_taxonomy, validate_gbif_occurrence, RejectionReason

In [18]:
# WikipediaPage: el contrato pivote (1 por species_key)
print('WikipediaPage (contrato pivote):')
for field_name, field in WikipediaPage.model_fields.items():
    print(f'  {field_name:20s} | {field.annotation}')

WikipediaPage (contrato pivote):
  species_key          | <class 'str'>
  title                | <class 'str'>
  resolved_via         | <class 'str'>
  extract              | <class 'str'>
  sections_text        | <class 'str'>
  page_url             | <class 'str'>
  is_disambiguation    | <class 'bool'>


In [19]:
print('GBIFTaxonomy (contrato taxonomía, 1 por species_key):')
for field_name, field in GBIFTaxonomy.model_fields.items():
    print(f'  {field_name:20s} | {field.annotation}')

GBIFTaxonomy (contrato taxonomía, 1 por species_key):
  species_key          | <class 'str'>
  usage_key            | <class 'int'>
  scientific_name      | <class 'str'>
  canonical_name       | <class 'str'>
  rank                 | <class 'str'>
  match_type           | <class 'str'>
  confidence           | <class 'int'>
  kingdom              | typing.Optional[str]
  phylum               | typing.Optional[str]
  class_name           | typing.Optional[str]


In [20]:
print('GBIFOccurrence (contrato ocurrencias, N por species_key):')
for field_name, field in GBIFOccurrence.model_fields.items():
    print(f'  {field_name:30s} | {field.annotation}')

GBIFOccurrence (contrato ocurrencias, N por species_key):
  gbif_id                        | <class 'int'>
  species_key                    | <class 'str'>
  decimal_latitude               | <class 'float'>
  decimal_longitude              | <class 'float'>
  country                        | typing.Optional[str]
  event_date                     | typing.Optional[datetime.datetime]
  basis_of_record                | <class 'str'>
  iucn_red_list_category         | typing.Optional[str]
  coordinate_uncertainty_m       | typing.Optional[float]


In [21]:
# Validar datos reales contra los contratos
tax_raw = fetch_gbif_taxonomy('Carcharodon carcharias')
result = validate_gbif_taxonomy({'raw': tax_raw}, batch_id='demo', species_key='carcharodon_carcharias')
if result:
    print('✅ GBIF Taxonomy válido:')
    print(f"   usage_key:      {result['usage_key']}")
    print(f"   canonical_name: {result['canonical_name']}")
    print(f"   rank:           {result['rank']}")
    print(f"   kingdom:        {result['kingdom']}")
    print(f"   phylum:         {result['phylum']}")
    print(f"   class_name:     {result['class_name']}")
    print(f"   match_type:     {result['match_type']}")
else:
    print('❌ Taxonomía rechazada')

✅ GBIF Taxonomy válido:
   usage_key:      2420694
   canonical_name: Carcharodon carcharias
   rank:           SPECIES
   kingdom:        Animalia
   phylum:         Chordata
   class_name:     Elasmobranchii
   match_type:     EXACT


In [22]:
# Validar ocurrencias reales
occ_raw = fetch_gbif_occurrences('Carcharodon carcharias')
records = occ_raw['data'].get('results', [])
valid = 0
rejected = 0
for rec in records:
    r = validate_gbif_occurrence(rec, batch_id='demo', species_key='carcharodon_carcharias')
    if r:
        valid += 1
    else:
        rejected += 1
print(f'Ocurrencias de Carcharodon carcharias:')
print(f'  Válidas:   {valid}')
print(f'  Rechazadas:{rejected}')
print(f'  Total:     {len(records)}')

Ocurrencias de Carcharodon carcharias:
  Válidas:   43
  Rechazadas:7
  Total:     50


In [23]:
# Motivos de rechazo disponibles
print('Motivos de rechazo:')
for r in RejectionReason:
    print(f'  {r.value}')

Motivos de rechazo:
  PAGE_NOT_FOUND
  AMBIGUOUS_RESOLUTION
  EXTRACT_TOO_SHORT
  DISAMBIGUATION_PAGE
  HTTP_ERROR
  MISSING_SECTIONS_TEXT
  MATCH_TYPE_NOT_EXACT
  NO_MATCH_RETURNED
  MISSING_COORDINATES
  COORDINATES_OUT_OF_RANGE
  MISSING_GBIF_ID
  INVALID_EVENT_DATE
  MISSING_BASIS_OF_RECORD


### 3.3 Merge Layer — Integración de conocimiento

Wikipedia es el **pivote mandatorio** (1:1). Si no hay Wikipedia válida, la especie no avanza.
GBIF Taxonomy se une solo si `match_type == 'EXACT'`; de lo contrario queda como `UNMATCHED`.
GBIF Occurrence se agrega (N → 1) en un resumen por especie con países, rango de años y categoría IUCN.

In [24]:
from src.merge.merger import merge_species, MergedSpecies, GBIFOccurrenceSummary

print('MergedSpecies — registro fusionado por species_key:')
for field_name, field in MergedSpecies.model_fields.items():
    print(f'  {field_name:25s} | {field.annotation}')

MergedSpecies — registro fusionado por species_key:
  species_key               | <class 'str'>
  wikipedia_title           | <class 'str'>
  extract                   | <class 'str'>
  sections_text             | <class 'str'>
  kingdom                   | typing.Optional[str]
  phylum                    | typing.Optional[str]
  class_name                | typing.Optional[str]
  rank                      | typing.Optional[str]
  taxonomy_status           | <class 'str'>
  n_occurrences_validas     | <class 'int'>
  paises                    | list[str]
  anio_min                  | typing.Optional[int]
  anio_max                  | typing.Optional[int]
  iucn_red_list_category    | typing.Optional[str]
  augmentation_status       | <class 'str'>


In [25]:
# Fusionar una especie con datos reales
sk = 'carcharodon_carcharias'

# 1. Validar Wikipedia
res = resolve_wikipedia_title('Carcharodon carcharias')
wp_summary = fetch_wikipedia_summary(res.title)
wp_extracts = fetch_wikipedia_extracts(res.title)
wp_valid = validate_wikipedia(
    {
        'raw': {'summary': wp_summary, 'extracts': wp_extracts},
        'resolved_via': res.resolved_via.value if res.resolved_via else 'SEARCH',
    },
    batch_id='demo',
    species_key=sk,
)

# 2. Validar GBIF Taxonomía
tax_raw = fetch_gbif_taxonomy('Carcharodon carcharias')
tax_valid = validate_gbif_taxonomy({'raw': tax_raw}, batch_id='demo', species_key=sk)

# 3. Validar ocurrencias
occ_raw = fetch_gbif_occurrences('Carcharodon carcharias')
occ_valid = []
for rec in occ_raw['data'].get('results', []):
    r = validate_gbif_occurrence(rec, batch_id='demo', species_key=sk)
    if r:
        occ_valid.append(r)

# 4. Fusionar
merged = merge_species(wp_valid, tax_valid, occ_valid)
if merged:
    print('✅ MergedSpecies generado:\n')
    print(json.dumps(merged.model_dump(), indent=2, default=str))

✅ MergedSpecies generado:

{
  "species_key": "carcharodon_carcharias",
  "wikipedia_title": "Great white shark",
  "extract": "The great white shark, also known as the white shark, white pointer, or great white, is a large shark. It is closely related to the mako sharks, the porbeagle, and the salmon shark. It is a robustly built species with a grayish upperside and a white underside. The white shark is one of the largest living shark and fish species, though it is still smaller than the whale shark and basking shark. It has about 300 triangular, serrated teeth that are continuously replaced. Its massive, fatty liver can reach over a quarter of its body weight, providing buoyancy and storing energy. The species is partially warm-blooded, an adaptation that allows it to remain active in colder waters.",
  "sections_text": "The great white shark (Carcharodon carcharias), also known as the white shark, white pointer, or great white, is a large shark. It is closely related to the mako sha

### 3.4 Warehouse Layer — DuckDB

Motor embebido **DuckDB** con 5 tablas, cada una con UPSERT idempotente por clave natural.

In [26]:
from src.warehouse.loader import Warehouse
# Cerrar conexiones DuckDB previas del mismo proceso
Warehouse.close_all()

with Warehouse() as wh:
    wh.initialize_schema()
    print('Tablas creadas en warehouse.duckdb:')
    tables = wh.connect().execute(
        "SELECT table_name FROM information_schema.tables WHERE table_schema = 'main'"
    ).fetchall()
    for t in tables:
        print(f'  - {t[0]}')

Tablas creadas en warehouse.duckdb:
  - silver_gbif_occurrences
  - silver_gbif_taxonomy
  - silver_wikipedia
  - species_catalog
  - species_merged


In [27]:
# Esquema de species_merged (la tabla principal de conocimiento)
with Warehouse() as wh:
    conn = wh.connect()
    schema = conn.execute('DESCRIBE species_merged').fetchall()
    print('Esquema de species_merged:')
    for col in schema:
        print(f'  {str(col[0]):30s} | {col[1]}')

Esquema de species_merged:
  species_key                    | VARCHAR
  wikipedia_title                | VARCHAR
  extract                        | VARCHAR
  sections_text                  | VARCHAR
  kingdom                        | VARCHAR
  phylum                         | VARCHAR
  class_name                     | VARCHAR
  rank                           | VARCHAR
  taxonomy_status                | VARCHAR
  n_occurrences_validas          | INTEGER
  paises                         | VARCHAR[]
  anio_min                       | INTEGER
  anio_max                       | INTEGER
  iucn_red_list_category         | VARCHAR
  augmentation_status            | VARCHAR
  updated_at                     | TIMESTAMP


In [28]:
# Esquema de species_catalog (registro creciente del universo de especies)
with Warehouse() as wh:
    conn = wh.connect()
    schema_cat = conn.execute('DESCRIBE species_catalog').fetchall()
    print('Esquema de species_catalog:')
    for col in schema_cat:
        print(f'  {str(col[0]):25s} | {col[1]}')

Esquema de species_catalog:
  species_key               | VARCHAR
  scientific_name           | VARCHAR
  wikipedia_title           | VARCHAR
  first_seen_batch_id       | VARCHAR
  first_processed_at        | TIMESTAMP
  last_processed_at         | TIMESTAMP
  origin                    | VARCHAR


### 3.5 Gold Layer — Conocimiento canónico + Búsqueda semántica

Tres entidades:

1. **Species (Parent)** — documento completo con 12 secciones canónicas.
2. **SpeciesChunk (Child)** — fragmentos semánticos por sección, cada uno con embedding.
3. **SpeciesMetadata** — filtros estructurados (taxonomía, IUCN, países, años).

Embeddings: `BAAI/bge-small-en-v1.5` (384 dim), FAISS IndexHNSWFlat.

In [29]:
from src.gold.models import Species, SpeciesChunk, SpeciesMetadata

print('Species — 12 secciones canónicas:')
for field_name, field in Species.model_fields.items():
    print(f'  {field_name:20s} | {field.annotation}')

Species — 12 secciones canónicas:
  species_key          | <class 'str'>
  identity             | typing.Optional[str]
  taxonomy             | typing.Optional[str]
  morphology           | typing.Optional[str]
  distribution         | typing.Optional[str]
  habitat              | typing.Optional[str]
  behavior             | typing.Optional[str]
  diet                 | typing.Optional[str]
  reproduction         | typing.Optional[str]
  conservation         | typing.Optional[str]
  human_interaction    | typing.Optional[str]
  references           | typing.Optional[str]
  multimedia           | typing.Optional[str]


In [30]:
print('SpeciesChunk — fragmento semántico embebido:')
for field_name, field in SpeciesChunk.model_fields.items():
    print(f'  {field_name:15s} | {field.annotation}')

SpeciesChunk — fragmento semántico embebido:
  chunk_id        | <class 'str'>
  species_key     | <class 'str'>
  section         | <class 'str'>
  text            | <class 'str'>


In [31]:
print('SpeciesMetadata — filtros estructurados:')
for field_name, field in SpeciesMetadata.model_fields.items():
    print(f'  {field_name:25s} | {field.annotation}')

SpeciesMetadata — filtros estructurados:
  species_key               | <class 'str'>
  kingdom                   | typing.Optional[str]
  phylum                    | typing.Optional[str]
  class_name                | typing.Optional[str]
  rank                      | typing.Optional[str]
  taxonomy_status           | <class 'str'>
  iucn_red_list_category    | typing.Optional[str]
  paises                    | list[str]
  anio_min                  | typing.Optional[int]
  anio_max                  | typing.Optional[int]
  n_occurrences_validas     | <class 'int'>
  augmentation_status       | <class 'str'>


In [32]:
from src.gold.indexer import build_species_from_merged, chunk_species, embed_chunks, build_faiss_index, search

print('Funciones del indexador:')
print('  build_species_from_merged(merged_dict) -> Species dict')
print('  chunk_species(species_dict)            -> list[SpeciesChunk]')
print('  embed_chunks(chunks)                   -> np.ndarray (384 dim)')
print('  build_faiss_index(embeddings)          -> faiss.Index')
print('  search(query, top_k, filtros)          -> list[dict]')
print()
# Demostrar chunking con datos ya obtenidos
if merged:
    species_dict = build_species_from_merged(merged.model_dump())
    chunks = chunk_species(species_dict)
    print(f'Species: {species_dict["species_key"]}')
    print(f'Secciones pobladas: {sum(1 for k, v in species_dict.items() if k != "species_key" and v is not None)}')
    print(f'Chunks generados:   {len(chunks)}')
    print()
    for ch in chunks:
        print(f'  {ch["chunk_id"]}')

Funciones del indexador:
  build_species_from_merged(merged_dict) -> Species dict
  chunk_species(species_dict)            -> list[SpeciesChunk]
  embed_chunks(chunks)                   -> np.ndarray (384 dim)
  build_faiss_index(embeddings)          -> faiss.Index
  search(query, top_k, filtros)          -> list[dict]

Species: carcharodon_carcharias
Secciones pobladas: 10
Chunks generados:   10

  carcharodon_carcharias__Identity__0
  carcharodon_carcharias__Taxonomy__0
  carcharodon_carcharias__Morphology__0
  carcharodon_carcharias__Distribution__0
  carcharodon_carcharias__Behavior__0
  carcharodon_carcharias__Diet__0
  carcharodon_carcharias__Reproduction__0
  carcharodon_carcharias__Conservation__0
  carcharodon_carcharias__Human Interaction__0
  carcharodon_carcharias__References__0


### 3.6 Búsqueda híbrida (Parent-Child Retrieval)

Flujo:
1. Embed query con `BAAI/bge-small-en-v1.5`
2. FAISS oversampling (`top_k * 5`)
3. Deduplicación por `species_key` (mejor score)
4. Filtros estructurados (taxonomía, IUCN, país)
5. Top-K resultados con metadatos

In [33]:
# Firma de search()
import inspect
sig = inspect.signature(search)
print('search() — parámetros:')
for name, param in sig.parameters.items():
    default = f' = {param.default}' if param.default is not inspect.Parameter.empty else ''
    print(f'  {name}: {param.annotation}{default}')

search() — parámetros:
  query: str
  top_k: int = 5
  filtro_taxonomia: Optional[dict] = None
  filtro_iucn: Optional[str] = None
  filtro_pais: Optional[str] = None


---
## Resumen

| Capa | Formato | Propósito |
|------|---------|-----------|
| **Bronze** | JSON crudo timestamped | Capturar todo tal cual de las APIs |
| **Silver** | JSON validado por Pydantic | Contratos, validación y cuarentena |
| **Merge** | Registro fusionado | Wikipedia + GBIF en un solo documento |
| **Warehouse** | DuckDB | Almacenamiento idempotente y catálogo de especies |
| **Gold** | FAISS + Parquet | Conocimiento canónico, chunks, embeddings y búsqueda |

---
*Shark Knowledge Medallion — Arquitectura Medallón para base de conocimiento de tiburones*